# Whisper Fine-Tuning for Tajik STT 

## Prepared by: Franz Kingstein Nesakumar

In [ ]:
# 🔐 Login to Hugging Face
from huggingface_hub import notebook_login
notebook_login()

## 1. Libraries

In [6]:
import torch

# Detect and set device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Move model to device
model.to(device)


Using device: cuda


WhisperForConditionalGeneration(
  (model): WhisperModel(
    (encoder): WhisperEncoder(
      (conv1): Conv1d(80, 768, kernel_size=(3,), stride=(1,), padding=(1,))
      (conv2): Conv1d(768, 768, kernel_size=(3,), stride=(2,), padding=(1,))
      (embed_positions): Embedding(1500, 768)
      (layers): ModuleList(
        (0-11): 12 x WhisperEncoderLayer(
          (self_attn): WhisperSdpaAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=False)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
        

In [4]:
from datasets import load_dataset, Audio
from transformers import WhisperProcessor, WhisperForConditionalGeneration, TrainingArguments, Trainer
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import numpy as np
import torch
import re
import evaluate

/home/franz/Documents/Internship/Code_1_stt/sst/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


##  1. Dataset

In [7]:
dataset = load_dataset("google/fleurs", "tg_tj")
sample = dataset["train"][0]

##  2.Load Tajik dataset from FLEURS

In [25]:
from datasets import load_dataset

# Load the Tajik dataset
dataset = load_dataset("google/fleurs", "tg_tj")

# Split the dataset into train and eval sets
train_dataset = dataset["train"]
eval_dataset = dataset["validation"]

## 3. Normalize Transcriptions

In [9]:
import re

def normalize(text):
    text = text.lower()
    text = re.sub(r"[^\w\s’ʼʼғқҷҳ]", "", text)  # remove special chars
    return text.strip()


## 4. Load Whisper model and processor

In [5]:
model_id = "openai/whisper-small"
processor = WhisperProcessor.from_pretrained(model_id)
model = WhisperForConditionalGeneration.from_pretrained(model_id)
model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language="tajik", task="transcribe")

## 5. Preprocess audio + text

In [30]:
def normalize(text):
    # Example normalization: strip whitespace and convert to lowercase
    return text.strip().lower()

In [45]:
def preprocess(batch):
    audio_list = batch["audio"]
    transcription_list = batch["transcription"]

    # Normalize transcription text
    normalized_texts = [normalize(t) for t in transcription_list]

    input_features = []
    labels = []

    for audio, text in zip(audio_list, normalized_texts):
        # Extract input features from audio
        inputs = processor(audio["array"], sampling_rate=16000, return_tensors="pt", padding=True)
        input_features.append(inputs.input_features[0].numpy())  # Use input_features

        # Tokenize text as labels (no as_target_processor)
        label_ids = processor.tokenizer(text).input_ids
        labels.append(label_ids)

    return {
        "input_features": input_features,
        "labels": labels,
    }



train_data = dataset["train"].map(
    preprocess,
    remove_columns=["audio", "transcription"],
    batched=True,
    batch_size=8,
    num_proc=4,
    desc="Preprocessing training dataset",
)

eval_data = dataset["validation"].map(
    preprocess,
    remove_columns=["audio", "transcription"],
    batched=True,
    batch_size=8,
    num_proc=4,
    desc="Preprocessing validation dataset",
)


Preprocessing validation dataset (num_proc=4): 100%|██████████| 240/240 [00:06<00:00, 34.34 examples/s]


## 6. Collator for variable-length labels

In [65]:
from torch.nn.utils.rnn import pad_sequence

class DataCollatorWhisper:
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, features):
        # Convert each input_features to tensor
        input_features = [torch.tensor(f["input_features"]) for f in features]
        # Pad sequences to the max length in batch on the time dimension (dim=1)
        input_features_padded = pad_sequence(input_features, batch_first=True, padding_value=0.0)

        # Convert labels and pad (assuming labels are 1D tensors)
        labels = [torch.tensor(f["labels"]) for f in features]
        labels_padded = pad_sequence(labels, batch_first=True, padding_value=-100)

        return {"input_features": input_features_padded, "labels": labels_padded}


In [63]:
def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # If pred_ids are logits, get the argmax
    if isinstance(pred_ids, tuple) or pred_ids.ndim == 3:
        pred_ids = pred_ids.argmax(axis=-1)

    # Convert to list if needed
    pred_ids = pred_ids.tolist() if hasattr(pred_ids, "tolist") else pred_ids
    label_ids = label_ids.tolist() if hasattr(label_ids, "tolist") else label_ids

    # Decode predictions
    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)

    # Replace -100 in labels with pad_token_id before decoding
    label_ids = [
        [token if token != -100 else processor.tokenizer.pad_token_id for token in seq]
        for seq in label_ids
    ]
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)

    # Compute WER
    wer = wer_metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer}


## 7. Training setup

In [51]:
# Keep your preprocess as-is (returns 'input_features' and 'labels')
# Use your custom DataCollatorWhisper to batch data correctly
data_collator = DataCollatorWhisper(processor=processor)

# TrainingArguments as you wrote are good; just make sure remove_unused_columns=False (which you added)
training_args = TrainingArguments(
    output_dir="./whisper-checkpoints",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    save_strategy="epoch",
    save_total_limit=1,
    fp16=True,
    logging_steps=10,
    num_train_epochs=3,
    report_to="none",
    remove_unused_columns=False,  # Important when model expects input_features and labels explicitly
)


In [33]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=processor.tokenizer, return_tensors="pt")

## 8. Train the model

In [66]:
import time
import os
from transformers import Trainer, TrainingArguments
from transformers.trainer_utils import get_last_checkpoint

# Start timer
start_time = time.time()

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,    # preprocessed train dataset
    eval_dataset=eval_data,      # preprocessed eval dataset
    data_collator=data_collator, # your custom data collator
    tokenizer=processor.tokenizer,
)

# Check if checkpoint exists and resume from it
checkpoint_dir = training_args.output_dir
resume_from = get_last_checkpoint(checkpoint_dir) if os.path.isdir(checkpoint_dir) else None

# Train the model (resume if checkpoint found)
trainer.train(resume_from_checkpoint=resume_from)

# End timer and print duration
end_time = time.time()
print(f"Training completed in {(end_time - start_time)/60:.2f} minutes")


/tmp/ipykernel_38704/328086621.py:10: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


RuntimeError: stack expects each tensor to be equal size, but got [80, 2400] at entry 0 and [80, 456] at entry 1

## 9.  Evaluate Word Error Rate (WER) (Optional)

In [ ]:
import evaluate
metric = evaluate.load("wer")

# Example test
preds = ["Салом, ман хубам"]
refs = ["Салом, ман хубам"]
print("WER:", metric.compute(predictions=preds, references=refs))

## 10. Export Your Model

In [ ]:
model.save_pretrained("./whisper-tgk")
processor.save_pretrained("./whisper-tgk")

In [ ]:
# 🚀 Push final model to the Hugging Face Hub
trainer.push_to_hub()
processor.push_to_hub('whisper-tajik-finetuned')